# Baseline 2 — TabTransformer on Mendeley URL features
**Mục tiêu:** 12 URL features (no HTML) → TabTransformer → binary classification  
**Dataset:** `mendeley-phishing-2021` → `index.csv` (không cần giải nén html)  
**Thời gian:** ~20–40 phút trên GPU T4

In [ ]:
import os, sys, json, re, math, warnings
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
)
from tqdm.notebook import tqdm

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})
warnings.filterwarnings('ignore')

SEED = 42; N_FOLDS = 5
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
KAGGLE_INPUT = Path('/kaggle/input')
PROJECT = Path('..')
RAW_DIR = PROJECT / 'data' / 'raw'
OUT_DIR = Path('/kaggle/working') if KAGGLE_INPUT.exists() else PROJECT / 'data'
MODEL_DIR = OUT_DIR / 'models'; FIG_DIR = OUT_DIR / 'figures'
MODEL_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)

# Auto-detect Mendeley index.csv anywhere under /kaggle/input/
mendeley_idx = RAW_DIR / 'mendeley' / 'index.csv'
if KAGGLE_INPUT.exists():
    for p in KAGGLE_INPUT.rglob('index.csv'):
        mendeley_idx = p; break
print(f'Mendeley idx: {mendeley_idx.exists()}')

In [ ]:
SUSPICIOUS_KEYWORDS = ['login','secure','verify','account','update','banking',
    'confirm','signin','password','reset','authenticate','paypal','webscr','free','bonus']
COMMON_TLDS = {'com','org','net','gov','edu','mil','io','co','uk',
               'au','de','jp','fr','ca','ru','cn','in','br','pl',
               'html','php','asp','jsp'}
URL_FEATURE_KEYS = ['url_length','domain_length','path_length','entropy',
    'special_char_ratio','digit_ratio','subdomain_count','has_https',
    'has_ip_address','suspicious_keywords','url_depth','tld_in_path']
TABULAR_DIM = 29

def shannon_entropy(text):
    if not text: return 0.0
    e, l = 0.0, len(text)
    for c in set(text):
        p = text.count(c) / l
        if p > 0: e -= p * math.log2(p)
    return round(e, 4)

def extract_url_features(url):
    parsed = urlparse(url)
    domain = (parsed.netloc or parsed.hostname or '').split(':')[0]
    path = parsed.path or ''
    fu = url.strip(); parts = domain.split('.')
    sc = sum(1 for c in fu if c in '@-_?.&=%+#~!'); dc = sum(1 for c in fu if c.isdigit())
    tc = max(len(fu), 1)
    ip_r = re.compile(r'^(?:(?:25[0-5]|2[0-4]\d|[01]?\d\d?)\.){3}(?:25[0-5]|2[0-4]\d|[01]?\d\d?)$')
    return {
        'url_length': len(fu), 'domain_length': len(domain),
        'path_length': len(path), 'entropy': shannon_entropy(fu),
        'special_char_ratio': round(sc/tc, 4), 'digit_ratio': round(dc/tc, 4),
        'subdomain_count': max(0, len(parts)-2) if len(parts) >= 2 else 0,
        'has_https': 1 if parsed.scheme == 'https' else 0,
        'has_ip_address': 1 if ip_r.match(domain) else 0,
        'suspicious_keywords': sum(1 for kw in SUSPICIOUS_KEYWORDS if kw in fu.lower()),
        'url_depth': len([s for s in path.split('/') if s]),
        'tld_in_path': 1 if any(f'.{t}' in path.lower() for t in COMMON_TLDS) else 0,
    }

In [ ]:
df = pd.read_csv(mendeley_idx, encoding='utf-8')
print(f'Records: {len(df)}')
print(f'Phishing: {(df["result"]==1).sum()}, Genuine: {(df["result"]==0).sum()}')

# 12 URL features + 17 DNS/WHOIS/SSL defaults = 29-dim
url_vectors = []
for _, row in tqdm(df.iterrows(), total=len(df), desc='Extracting URL features'):
    feats = extract_url_features(str(row['url']).strip())
    vec = [feats[k] for k in URL_FEATURE_KEYS]
    # Pad with DNS/WHOIS/SSL defaults (17 features, all -1)
    vec += [-1.0] * (TABULAR_DIM - len(vec))
    url_vectors.append(vec)

X = np.array(url_vectors, dtype=np.float32)

# StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X).astype(np.float32)

y = df['result'].values.astype(np.float32)
print(f'Shape: {X.shape}, Phishing: {y.sum()}, Benign: {(y==0).sum()}')


In [ ]:
class FeatureEmbedding(nn.Module):
    def __init__(self, d=32): super().__init__(); self.e = nn.Linear(1, d)
    def forward(self, x): return self.e(x.unsqueeze(-1))

class TabTransformer(nn.Module):
    def __init__(self, nf=29, ed=32, nh=4, hd=256, od=128, dp=0.1):
        super().__init__()
        self.embs = nn.ModuleList([FeatureEmbedding(ed) for _ in range(nf)])
        self.attn = nn.MultiheadAttention(ed, nh, batch_first=True, dropout=dp)
        self.n1 = nn.LayerNorm(ed); self.n2 = nn.LayerNorm(ed)
        self.ff = nn.Sequential(nn.Linear(ed, hd), nn.GELU(), nn.Dropout(dp), nn.Linear(hd, ed), nn.Dropout(dp))
        self.proj = nn.Linear(ed * nf, od)
        self.cls = nn.Sequential(nn.Linear(od, 64), nn.ReLU(), nn.Dropout(dp), nn.Linear(64, 1))
    def forward(self, x):
        h = torch.stack([e(x[:, i]) for i, e in enumerate(self.embs)], 1)
        a, _ = self.attn(h, h, h); h = self.n1(h + a)
        f = self.ff(h); h = self.n2(h + f)
        return self.cls(self.proj(h.reshape(h.size(0), -1)))

In [ ]:
class SimpleDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X) if isinstance(X, np.ndarray) else X
        self.y = torch.from_numpy(y.reshape(-1,1).astype(np.float32)) if isinstance(y, np.ndarray) else y
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def compute_metrics(labels, preds):
    pb = (preds >= 0.5).astype(int)
    fpr_val = 0.0
    if len(np.unique(labels)) > 1:
        cm  = confusion_matrix(labels, pb)
        tn, fp = cm[0, 0], cm[0, 1]
        fpr_val = round(fp / max(tn + fp, 1), 4)
    return {'accuracy': accuracy_score(labels, pb),
            'precision': precision_score(labels, pb, zero_division=0),
            'recall': recall_score(labels, pb, zero_division=0),
            'f1': f1_score(labels, pb, zero_division=0),
            'auc': roc_auc_score(labels, preds) if len(np.unique(labels)) > 1 else 0.0,
            'fpr': fpr_val}

def train_epoch(model, loader, opt, crit):
    model.train(); total = 0
    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad(); loss = crit(model(Xb), yb)
        loss.backward(); opt.step(); total += loss.item() * Xb.size(0)
    return total / len(loader.dataset)

def evaluate(model, loader, crit):
    model.eval(); total = 0; preds, labs = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            logits = model(Xb)
            total += crit(logits, yb).item() * Xb.size(0)
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            labs.extend(yb.cpu().numpy())
    preds, labs = np.array(preds), np.array(labs)
    m = compute_metrics(labs, preds); m['loss'] = total / len(loader.dataset)
    return m, preds, labs

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
all_metrics, all_preds, all_labels = [], [], []
BS, EP, LR = 128, 50, 1e-3

for fold, (tr_idx, te_idx) in enumerate(skf.split(X, y)):
    print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')
    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]
    tr_ld = DataLoader(SimpleDataset(X_tr, y_tr), batch_size=BS, shuffle=True)
    te_ld = DataLoader(SimpleDataset(X_te, y_te), batch_size=BS)
    model = TabTransformer(nf=X.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EP)
    crit = nn.BCEWithLogitsLoss()
    for ep in range(1, EP + 1):
        tl = train_epoch(model, tr_ld, opt, crit)
        m, _, _ = evaluate(model, te_ld, crit)
        sched.step()
        if ep % 10 == 0:
            print(f'  Epoch {ep:2d}/{EP} | Loss: {tl:.4f} | AUC: {m["auc"]:.4f} | F1: {m["f1"]:.4f}')
    fm, fp, fl = evaluate(model, te_ld, crit)
    fm['fold'] = fold + 1
    all_metrics.append(fm); all_preds.append(fp); all_labels.append(fl)
    torch.save(model.state_dict(), MODEL_DIR / f'baseline2_fold{fold+1}.pt')
    print(f'  Done: Acc={fm["accuracy"]:.4f}, AUC={fm["auc"]:.4f}, F1={fm["f1"]:.4f}')

avg = {k: np.mean([m[k] for m in all_metrics]) for k in ['accuracy','precision','recall','f1','auc','fpr']}
std = {k: np.std([m[k] for m in all_metrics]) for k in ['accuracy','precision','recall','f1','auc','fpr']}
print(f'\n>>> 5-Fold CV: Acc={avg["accuracy"]:.4f}+-{std["accuracy"]:.4f}, AUC={avg["auc"]:.4f}+-{std["auc"]:.4f}, F1={avg["f1"]:.4f}+-{std["f1"]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00']

all_p = np.concatenate(all_preds); all_l = np.concatenate(all_labels)
cm = confusion_matrix(all_l, (all_p >= 0.5).astype(int))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', ax=axes[0],
            xticklabels=['Benign','Phishing'], yticklabels=['Benign','Phishing'])
axes[0].set_title('Baseline 2 — Confusion Matrix'); axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

for f in range(N_FOLDS):
    fpr, tpr, _ = roc_curve(all_labels[f], all_preds[f])
    axes[1].plot(fpr, tpr, color=colors[f], lw=1.5, alpha=0.7,
                 label=f'Fold {f+1} (AUC={roc_auc_score(all_labels[f], all_preds[f]):.4f})')
fpr, tpr, _ = roc_curve(all_l, all_p)
axes[1].plot(fpr, tpr, 'k--', lw=2.5, label=f'Mean (AUC={roc_auc_score(all_l, all_p):.4f})')
axes[1].plot([0,1],[0,1], 'gray', lw=1, alpha=0.5)
axes[1].set_title('ROC Curves'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend(fontsize=8, loc='lower right')

names = ['accuracy','precision','recall','f1','auc']
x = np.arange(len(names)); means = [avg[m] for m in names]; stdevs = [std[m] for m in names]
axes[2].bar(x, means, yerr=stdevs, capsize=5, color='#e41a1c', alpha=0.8)
axes[2].set_xticks(x); axes[2].set_xticklabels([m.capitalize() for m in names])
axes[2].set_ylim(0, 1); axes[2].set_title('Metrics (Mean+-Std)')
for i, (m, s) in enumerate(zip(means, stdevs)):
    axes[2].text(i, m + s + 0.02, f'{m:.3f}+-{s:.3f}', ha='center', fontsize=8)

plt.tight_layout(); plt.savefig(FIG_DIR / 'baseline2_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "baseline2_summary.png"}')

In [ ]:
results = {'model': 'Baseline 2 - TabTransformer (Mendeley URL)', **avg, **{k + '_std': float(std[k]) for k in std}}
with open(MODEL_DIR / 'evaluation_baseline2.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {MODEL_DIR / "evaluation_baseline2.json"}')

---
### Download từ Output tab:
- `figures/baseline2_summary.png`
- `data/models/evaluation_baseline2.json`  
- `data/models/baseline2_fold1..5.pt`

Sau đó chạy **Proposed Model** → `kaggle_proposed.ipynb`